# Drive 데이터로 전체 실험 입력 준비

이 노트북은 모델을 실행하지 않고 기존 [`pilot_samples.json`](pilot_results/pilot_samples.json)과 Drive `dataset` 폴더의 한국어·영어 `train`/`validation`/`test` CSV **7개 파일**을 받아 하나의 `drive_study_inputs_v1.json`을 만듭니다. Colab 기본 `transformers`를 사용하며 Drive 마운트는 필요하지 않습니다. 업로드 파일명이 `(2)`처럼 바뀌어도 내용과 원래 이름을 확인합니다.

- 기존 보정: 예비 실험의 영어·한국어 128개 문서를 같은 토크나이저로 최대 512토큰까지 인코딩합니다.
- Drive 보정: `train`에서 문장 중복을 제거하고, 문서 사이에 EOS를 넣어 언어별 **128개 × 512토큰 구간**을 만듭니다. 기존의 128개 긴 문서와 표본 구성 방식이 다르므로 결과 해석 때 구분합니다.
- 후보 탐색: `validation`에서 언어별 3개 × 8192토큰을 만듭니다.
- 최종 평가: `test`에서 별도의 언어별 3개 × 8192토큰을 만듭니다.

기존 예비 실험의 보정·평가 텍스트와 정확히 겹치는 문장을 제외합니다. 우선순위는 `test → validation → train`이며, 각 분할 내부의 중복과 분할 간 중복도 제거합니다. 공백 정규화로 같은 문장을 찾으므로 비슷한 문장이나 같은 기사의 다른 구절까지 제거하지는 못합니다. 원본 CSV는 출력 JSON에 포함하지 않고 파일 해시와 사용한 토큰 ID만 저장합니다.


In [ ]:
import csv
import hashlib
import io
import json
import random
import re
from pathlib import Path

from google.colab import files
import transformers
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 22
N_GROUPS = 3
GROUP_TOKENS = 16 * 512
CALIBRATION_SAMPLES = 128
CALIBRATION_TOKENS = 512
EXPECTED_PILOT_SHA256 = "762e9a33457824598b73e49812666c3fb042f25df24a09db094ffc2a04590ed6"
EXPECTED_FILES = {"pilot_samples.json"} | {
    f"{lang}_{split}.csv"
    for lang in ("korean", "english")
    for split in ("train", "validation", "test")
}

uploaded = files.upload()
inputs = {}
for filename, data in uploaded.items():
    canonical = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", Path(filename).name)
    if canonical in inputs:
        raise ValueError(f"중복 업로드 파일: {canonical}")
    inputs[canonical] = data
if set(inputs) != EXPECTED_FILES:
    raise ValueError(f"정확히 이 7개 파일을 업로드하세요: {sorted(EXPECTED_FILES)}")
source_hashes = {name: hashlib.sha256(data).hexdigest() for name, data in inputs.items()}
if source_hashes["pilot_samples.json"] != EXPECTED_PILOT_SHA256:
    raise ValueError("저장된 예비 실험의 pilot_samples.json과 다릅니다.")
pilot = json.loads(inputs["pilot_samples.json"].decode("utf-8"))
if any(len(pilot[f"calibration_{lang}"]) != 128 or
       len(pilot["eval_groups"][lang]) != 3 for lang in ("ko", "en")):
    raise ValueError("예비 실험의 보정·평가 표본 구조가 예상과 다릅니다.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
print("입력 파일 7개와 예비 표본 해시 확인 완료")


In [ ]:
def text_key(text):
    return " ".join(text.split())

def clean_csv(data, forbidden):
    reader = csv.DictReader(io.TextIOWrapper(io.BytesIO(data), encoding="utf-8-sig"))
    if not reader.fieldnames or "text" not in reader.fieldnames:
        raise ValueError("CSV에 text 열이 필요합니다.")
    kept, seen = [], set()
    counts = {"rows": 0, "empty": 0, "prior_or_cross_split": 0, "within_split_duplicate": 0}
    for row in reader:
        counts["rows"] += 1
        text = row["text"].strip() if isinstance(row["text"], str) else ""
        key = text_key(text)
        if not key:
            counts["empty"] += 1
        elif key in forbidden:
            counts["prior_or_cross_split"] += 1
        elif key in seen:
            counts["within_split_duplicate"] += 1
        else:
            seen.add(key)
            kept.append(text)
    counts["kept"] = len(kept)
    return kept, seen, counts

cleaned, split_stats = {}, {}
for lang, stem in (("ko", "korean"), ("en", "english")):
    prior = {text_key(text) for text in pilot[f"calibration_{lang}"]}
    prior.update(text_key(text) for group in pilot["eval_groups"][lang] for text in group)
    forbidden = prior
    for split in ("test", "validation", "train"):
        name = f"{stem}_{split}.csv"
        texts, keys, counts = clean_csv(inputs[name], forbidden)
        cleaned[lang, split] = texts
        split_stats[lang, split] = counts
        forbidden = forbidden | keys
        print(lang, split, counts)


In [ ]:
def evaluation_groups(texts, seed):
    docs = list(texts)
    random.Random(seed).shuffle(docs)
    groups = [[] for _ in range(N_GROUPS)]
    used_docs = [0] * N_GROUPS
    for text in docs:
        index = min(range(N_GROUPS), key=lambda i: len(groups[i]))
        groups[index].extend(tokenizer(text, add_special_tokens=False)["input_ids"])
        if tokenizer.eos_token_id is not None:
            groups[index].append(tokenizer.eos_token_id)
        used_docs[index] += 1
        if all(len(group) >= GROUP_TOKENS for group in groups):
            break
    if any(len(group) < GROUP_TOKENS for group in groups):
        raise ValueError(f"평가 토큰 부족: {[len(group) for group in groups]}")
    return [group[:GROUP_TOKENS] for group in groups], used_docs

def drive_calibration(texts, seed):
    docs = list(texts)
    random.Random(seed).shuffle(docs)
    samples, buffer, used_docs = [], [], 0
    for text in docs:
        buffer.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
        if tokenizer.eos_token_id is not None:
            buffer.append(tokenizer.eos_token_id)
        used_docs += 1
        while len(buffer) >= CALIBRATION_TOKENS and len(samples) < CALIBRATION_SAMPLES:
            samples.append(buffer[:CALIBRATION_TOKENS])
            buffer = buffer[CALIBRATION_TOKENS:]
        if len(samples) == CALIBRATION_SAMPLES:
            return samples, used_docs
    raise ValueError(f"Drive train의 보정 토큰이 부족합니다: {len(samples)}개")

calibration = {}
for lang in ("ko", "en"):
    calibration[f"pilot_{lang}"] = [
        tokenizer(text, truncation=True, max_length=CALIBRATION_TOKENS)["input_ids"]
        for text in pilot[f"calibration_{lang}"]
    ]
    calibration[f"drive_{lang}"], used = drive_calibration(
        cleaned[lang, "train"], SEED + (10 if lang == "ko" else 11)
    )
    split_stats[lang, "train"]["calibration_docs_consumed"] = used

validation, test = {}, {}
for lang in ("ko", "en"):
    validation[lang], used = evaluation_groups(
        cleaned[lang, "validation"], SEED + (20 if lang == "ko" else 21)
    )
    split_stats[lang, "validation"]["eval_docs_consumed"] = used
    test[lang], used = evaluation_groups(
        cleaned[lang, "test"], SEED + (30 if lang == "ko" else 31)
    )
    split_stats[lang, "test"]["eval_docs_consumed"] = used

assert all(len(calibration[f"pilot_{lang}"]) == CALIBRATION_SAMPLES and
           len(calibration[f"drive_{lang}"]) == CALIBRATION_SAMPLES
           for lang in ("ko", "en"))
assert all(len(group) == GROUP_TOKENS for split in (validation, test)
           for groups in split.values() for group in groups)
print("보정 4세트와 validation/test 평가 묶음 생성 완료")


In [ ]:
payload = {
    "format": "team22_drive_study_v1",
    "model_id": MODEL_ID,
    "seed": SEED,
    "transformers": transformers.__version__,
    "tokenizer_sha256": hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode()).hexdigest(),
    "source_sha256": source_hashes,
    "split_stats": {f"{lang}_{split}": counts for (lang, split), counts in split_stats.items()},
    "calibration": calibration,
    "validation": validation,
    "test": test,
}
output_name = "drive_study_inputs_v1.json"
with open(output_name, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, separators=(",", ":"))
print("출력:", output_name, "SHA-256:", hashlib.sha256(Path(output_name).read_bytes()).hexdigest())
files.download(output_name)
